# Experiment A3 reproduction

A3 studies the iteration count of the block Lewis fixed-point iteration on 50 Gaussian $100\times20$ matrices, each partitioned into 20 consecutive blocks of five rows.

Running this notebook from the public `A3/` directory will:

1. verify the released saved data;
2. regenerate the matrix bank and block diagnostics from seeds 4001--4050;
3. rerun all 750 matrix/$p$ cases using the released numerical implementation;
4. recompute the primary and threshold summaries;
5. compare the fresh rerun with the released final results; and
6. regenerate the two-panel A3 figure in `reproduced/`.

The notebook never writes into `data/` or `figures/`. Fresh outputs go only to `reproduced/`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_triangular

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    raise RuntimeError("Run A3_reproduce.ipynb from the public A3 directory.")

DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures"
REPRO_DIR = ROOT / "reproduced"
REPRO_DIR.mkdir(exist_ok=True)

N = 20
N_BLOCKS = 20
ROWS_PER_BLOCK = 5
M = N_BLOCKS * ROWS_PER_BLOCK
DEV_SEED = 4000
FINAL_SEEDS = np.arange(4001, 4051)
P_GRID = np.array([
    3, 4, 5.7, 7.3, 10.8, 14.2, 19.6, 27.9,
    32, 38.4, 54.1, 64, 74.8, 98.6, 127.3,
], dtype=float)
THRESHOLDS = (1e-4, 1e-6, 1e-8)
PRIMARY_TOL = 1e-8
MAX_UPDATES = 10_000
BLOCK_STARTS = np.arange(0, M, ROWS_PER_BLOCK)
BLOCK_ENDS = BLOCK_STARTS + ROWS_PER_BLOCK

print(f"A3: {len(FINAL_SEEDS)} matrices x {len(P_GRID)} p-values = {len(FINAL_SEEDS)*len(P_GRID)} runs")
print(f"matrix shape: {M} x {N}; blocks: {N_BLOCKS} x {ROWS_PER_BLOCK} rows")


## Released data

The released bank contains the actual final matrices and block boundaries. The raw run summary contains the three threshold crossings, terminal diagnostics, and all 20 terminal block weights for each run. The trajectory file contains every saved scalar residual.


In [ ]:
with np.load(DATA_DIR / "A3_block_instance_bank.npz") as z:
    saved_matrices = z["matrices"].copy()
    saved_seeds = z["seeds"].copy()
    saved_block_starts = z["block_starts"].copy()
    saved_block_ends = z["block_ends"].copy()
    saved_m = int(z["m"])
    saved_n = int(z["n"])
    saved_k = int(z["k"])
    saved_rows_per_block = int(z["rows_per_block"])

saved_diag = pd.read_csv(DATA_DIR / "A3_block_instance_diagnostics.csv")
saved_run = pd.read_csv(DATA_DIR / "A3_run_summary.csv")
saved_traj = pd.read_csv(DATA_DIR / "A3_residual_trajectories.csv")
saved_primary = pd.read_csv(DATA_DIR / "A3_primary_summary_v2.csv")
saved_tol = pd.read_csv(DATA_DIR / "A3_tolerance_summary.csv")

assert saved_matrices.shape == (50, 100, 20)
assert np.array_equal(saved_seeds, FINAL_SEEDS)
assert np.array_equal(saved_block_starts, BLOCK_STARTS)
assert np.array_equal(saved_block_ends, BLOCK_ENDS)
assert (saved_m, saved_n, saved_k, saved_rows_per_block) == (100, 20, 20, 5)
assert len(saved_run) == 750
assert saved_run["converged"].all()
assert saved_run[["matrix_id", "p_index"]].drop_duplicates().shape[0] == 750
assert np.allclose(saved_primary["p"], P_GRID)
assert np.allclose(saved_tol["p"], P_GRID)

print("released bank and result structure: OK")
print("released residual rows:", len(saved_traj))
print("released T_1e-8 range:", int(saved_run.T_1e_8.min()), "to", int(saved_run.T_1e_8.max()))


## Matrix generator and block diagnostics

The final matrices are generated by `np.random.default_rng(seed).standard_normal((100, 20))`. Blocks are the consecutive row ranges $0{:}5,5{:}10,\ldots,95{:}100$.

Every final block has rank 5, so $R=\sum\_i r\_i=100$ and the general initialization $w\_i^{(0)}=nr\_i/R$ reduces to $w\_i^{(0)}=1$ for all 20 blocks.


In [ ]:
def make_a3_instance(seed):
    return np.random.default_rng(int(seed)).standard_normal((M, N))

regen_matrices = np.stack([make_a3_instance(seed) for seed in FINAL_SEEDS])
assert np.array_equal(regen_matrices, saved_matrices)

rows = []
for seed, A in zip(FINAL_SEEDS, regen_matrices):
    row = {
        "seed": int(seed),
        "rank": int(np.linalg.matrix_rank(A)),
        "kappa_2": float(np.linalg.cond(A)),
    }
    for i, (start, end) in enumerate(zip(BLOCK_STARTS, BLOCK_ENDS)):
        row[f"block_rank_{i:02d}"] = int(np.linalg.matrix_rank(A[start:end]))
    rows.append(row)
regen_diag = pd.DataFrame(rows)

assert np.array_equal(regen_diag["seed"].to_numpy(), saved_diag["seed"].to_numpy())
assert np.array_equal(regen_diag["rank"].to_numpy(), saved_diag["rank"].to_numpy())
rank_cols = [f"block_rank_{i:02d}" for i in range(N_BLOCKS)]
assert np.array_equal(regen_diag[rank_cols].to_numpy(), saved_diag[rank_cols].to_numpy())
assert np.allclose(regen_diag["kappa_2"], saved_diag["kappa_2"], rtol=0, atol=5e-15)

with open(REPRO_DIR / "A3_block_instance_bank.npz", "wb") as f:
    np.savez_compressed(
        f,
        matrices=regen_matrices,
        seeds=np.asarray(FINAL_SEEDS, dtype=int),
        block_starts=np.asarray(BLOCK_STARTS, dtype=int),
        block_ends=np.asarray(BLOCK_ENDS, dtype=int),
        m=np.asarray(M), n=np.asarray(N), k=np.asarray(N_BLOCKS),
        rows_per_block=np.asarray(ROWS_PER_BLOCK),
    )
regen_diag.to_csv(REPRO_DIR / "A3_block_instance_diagnostics.csv", index=False)

print("matrix bank regenerates exactly: True")
print("all stacked ranks = 20:", bool((regen_diag["rank"] == 20).all()))
print("all 1000 block ranks = 5:", bool((regen_diag[rank_cols].to_numpy() == 5).all()))
print("kappa_2(A) min/median/max:", *(f"{x:.6f}" for x in [regen_diag.kappa_2.min(), regen_diag.kappa_2.median(), regen_diag.kappa_2.max()]))


## Block Lewis update and stopping rule

For $r=1-2/p$, the implementation forms the row-scaled matrix $B=W^{r/2}A=QR$, solves $R^\top X^\top=A^\top$ for $X=AR^{-1}$, and updates each block by $w\_i^+=w\_i^r\lVert X\_i\rVert\_F^2$. Thus the row leverage contributions are summed within each five-row block.

Progress is measured by $R\_t=\max\_i\left|\log(w\_i^{(t+1)}/w\_i^{(t)})\right|$. We record the first strict crossings of $10^{-4}$, $10^{-6}$, and $10^{-8}$, with $10^{-8}$ as the run endpoint and a 10,000-iteration limit.


In [ ]:
def block_update_qr(A, w, p):
    r = 1.0 - 2.0 / p
    row_scale = np.repeat(w ** (r / 2.0), ROWS_PER_BLOCK)
    B = row_scale[:, None] * A
    _, R = np.linalg.qr(B, mode="reduced")
    X = solve_triangular(R.T, A.T, lower=True, check_finite=False).T
    X_blocks = X.reshape(N_BLOCKS, ROWS_PER_BLOCK, N)
    frob_sq = np.sum(X_blocks ** 2, axis=(1, 2))
    return (w ** r) * frob_sq


def run_block_lewis(A, p, max_updates=MAX_UPDATES):
    w = np.ones(N_BLOCKS, dtype=float)
    residuals = []
    crossings = {tau: None for tau in THRESHOLDS}

    for update in range(1, max_updates + 1):
        w_new = block_update_qr(A, w, p)
        if not np.all(np.isfinite(w_new)) or not np.all(w_new > 0):
            raise FloatingPointError("Invalid block weight encountered.")
        residual = float(np.max(np.abs(np.log(w_new / w))))
        residuals.append(residual)
        for tau in THRESHOLDS:
            if crossings[tau] is None and residual < tau:
                crossings[tau] = update
        w = w_new
        if residual < PRIMARY_TOL:
            break

    return {
        "converged": residuals[-1] < PRIMARY_TOL,
        "updates_run": len(residuals),
        "T_1e_4": crossings[1e-4],
        "T_1e_6": crossings[1e-6],
        "T_1e_8": crossings[1e-8],
        "final_residual": residuals[-1],
        "sum_defect": abs(np.sum(w) - N),
        "min_terminal_w": float(np.min(w)),
        "max_terminal_w": float(np.max(w)),
        "terminal_w": w.copy(),
        "residuals": np.asarray(residuals, dtype=float),
    }


## Verify the released analysis from released raw data

This step is independent of a fresh numerical rerun. It reconstructs all threshold crossings from the released residual trajectories and recomputes the two released summary tables.


In [ ]:
def first_crossing(residuals, tau):
    hits = np.flatnonzero(np.asarray(residuals) < tau)
    return -1 if len(hits) == 0 else int(hits[0] + 1)

cross_rows = []
for (matrix_id, p_index), g in saved_traj.groupby(["matrix_id", "p_index"], sort=False):
    g = g.sort_values("update")
    residuals = g["residual"].to_numpy()
    cross_rows.append({
        "matrix_id": int(matrix_id),
        "p_index": int(p_index),
        "T_1e_4_re": first_crossing(residuals, 1e-4),
        "T_1e_6_re": first_crossing(residuals, 1e-6),
        "T_1e_8_re": first_crossing(residuals, 1e-8),
        "updates_re": len(residuals),
    })
cross = pd.DataFrame(cross_rows)
check = saved_run.merge(cross, on=["matrix_id", "p_index"], validate="one_to_one")
for old, new in [("T_1e_4", "T_1e_4_re"), ("T_1e_6", "T_1e_6_re"), ("T_1e_8", "T_1e_8_re")]:
    assert np.array_equal(check[old].to_numpy(), check[new].to_numpy())
assert np.array_equal(check["updates_run"].to_numpy(), check["updates_re"].to_numpy())

primary_from_raw = (
    saved_run.groupby("p", sort=True)["T_1e_8"]
    .agg(
        n="count", mean="mean", std="std",
        q20=lambda x: x.quantile(0.20), median="median",
        q80=lambda x: x.quantile(0.80), min="min", max="max",
    ).reset_index()
)
tol_from_raw = (
    saved_run.groupby("p", sort=True)[["T_1e_4", "T_1e_6", "T_1e_8"]]
    .median().reset_index()
)

assert np.allclose(primary_from_raw[saved_primary.columns], saved_primary, rtol=0, atol=1e-12)
assert np.allclose(tol_from_raw[saved_tol.columns], saved_tol, rtol=0, atol=1e-12)

print("released threshold crossings reconstructed: 2250/2250")
print("released primary summary reproduced: True")
print("released threshold summary reproduced: True")


## Fresh rerun of all 750 cases

The fresh run uses regenerated matrices, the same block update, fresh all-ones initialization for every matrix/$p$ pair, and the same first-crossing rule. Small floating-point differences can occur across numerical-library environments, so the comparison below reports scientific agreement rather than requiring byte-identical trajectories.


In [ ]:
run_rows = []
traj_rows = []

for matrix_id, (seed, A) in enumerate(zip(FINAL_SEEDS, regen_matrices), start=1):
    kappa = float(regen_diag.loc[regen_diag.seed == seed, "kappa_2"].iloc[0])
    for p_index, p in enumerate(P_GRID, start=1):
        out = run_block_lewis(A, float(p))
        row = {
            "matrix_id": matrix_id,
            "seed": int(seed),
            "p_index": p_index,
            "p": float(p),
            "r": 1.0 - 2.0 / float(p),
            "kappa_2_A": kappa,
            "T_1e_4": -1 if out["T_1e_4"] is None else int(out["T_1e_4"]),
            "T_1e_6": -1 if out["T_1e_6"] is None else int(out["T_1e_6"]),
            "T_1e_8": -1 if out["T_1e_8"] is None else int(out["T_1e_8"]),
            "updates_run": int(out["updates_run"]),
            "final_residual": float(out["final_residual"]),
            "sum_defect": float(out["sum_defect"]),
            "min_terminal_w": float(out["min_terminal_w"]),
            "max_terminal_w": float(out["max_terminal_w"]),
            "finite_valid": True,
            "positive_valid": True,
            "converged": bool(out["converged"]),
            "status": "converged" if out["converged"] else "capped",
        }
        for i, wi in enumerate(out["terminal_w"]):
            row[f"terminal_w_{i:02d}"] = float(wi)
        run_rows.append(row)
        for update, residual in enumerate(out["residuals"], start=1):
            traj_rows.append({
                "matrix_id": matrix_id, "seed": int(seed), "p_index": p_index,
                "p": float(p), "update": update, "residual": float(residual),
            })

fresh_run = pd.DataFrame(run_rows).sort_values(["matrix_id", "p_index"]).reset_index(drop=True)
fresh_traj = pd.DataFrame(traj_rows).sort_values(["matrix_id", "p_index", "update"]).reset_index(drop=True)

assert len(fresh_run) == 750
assert fresh_run["converged"].all()
assert fresh_run["T_1e_8"].min() > 0

fresh_run.to_csv(REPRO_DIR / "A3_run_summary.csv", index=False)
fresh_traj.to_csv(REPRO_DIR / "A3_residual_trajectories.csv", index=False)

print("fresh runs complete:", len(fresh_run))
print("all fresh runs converged:", bool(fresh_run.converged.all()))
print("fresh residual rows:", len(fresh_traj))


In [ ]:
fresh_primary = (
    fresh_run.groupby("p", sort=True)["T_1e_8"]
    .agg(
        n="count", mean="mean", std="std",
        q20=lambda x: x.quantile(0.20), median="median",
        q80=lambda x: x.quantile(0.80), min="min", max="max",
    ).reset_index()
)
fresh_tol = (
    fresh_run.groupby("p", sort=True)[["T_1e_4", "T_1e_6", "T_1e_8"]]
    .median().reset_index()
)
fresh_primary.to_csv(REPRO_DIR / "A3_primary_summary_v2.csv", index=False)
fresh_tol.to_csv(REPRO_DIR / "A3_tolerance_summary.csv", index=False)

cmp = saved_run[["matrix_id", "p_index", "T_1e_4", "T_1e_6", "T_1e_8"]].merge(
    fresh_run[["matrix_id", "p_index", "T_1e_4", "T_1e_6", "T_1e_8"]],
    on=["matrix_id", "p_index"], suffixes=("_saved", "_fresh"), validate="one_to_one"
)
for tau in ["1e_4", "1e_6", "1e_8"]:
    cmp[f"d_{tau}"] = cmp[f"T_{tau}_fresh"] - cmp[f"T_{tau}_saved"]

max_count_diff = int(cmp[["d_1e_4", "d_1e_6", "d_1e_8"]].abs().to_numpy().max())
max_primary_median_diff = float(np.max(np.abs(fresh_primary["median"] - saved_primary["median"])))
max_primary_q_diff = float(np.max(np.abs(
    fresh_primary[["q20", "q80"]].to_numpy() - saved_primary[["q20", "q80"]].to_numpy()
)))

terminal_cols = [f"terminal_w_{i:02d}" for i in range(N_BLOCKS)]
saved_w = saved_run[terminal_cols].to_numpy()
fresh_w = fresh_run[terminal_cols].to_numpy()
max_terminal_log_diff = float(np.max(np.abs(np.log(fresh_w / saved_w))))

print("fresh-versus-saved maximum threshold-count difference:", max_count_diff)
print("fresh-versus-saved maximum primary-median difference:", max_primary_median_diff)
print("fresh-versus-saved maximum q20/q80 difference:", max_primary_q_diff)
print("fresh-versus-saved maximum terminal-weight log-relative difference:", f"{max_terminal_log_diff:.3e}")

if max_count_diff == 0:
    print("fresh stopping times match the released results exactly in this environment")
elif max_count_diff <= 2 and max_primary_median_diff <= 2:
    print("fresh results show only small environment-dependent floating-point differences")
else:
    warnings.warn("Fresh rerun differs materially from the released iteration counts; inspect the numerical environment.")


## Regenerate the paper figure

The released paper figure uses the median, 20th percentile, and 80th percentile of $T\_{10^{-8}}$ in the left panel and the median iteration count at all three thresholds in the right panel. The plotting code below follows the final flat two-panel layout used in the paper.


In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.55))
p = fresh_primary["p"].to_numpy()

ax = axes[0]
ax.plot(p, fresh_primary["median"], marker="o", linewidth=1.8, markersize=4, label="Median")
ax.plot(p, fresh_primary["q20"], linestyle="--", linewidth=1.4, label="20th percentile")
ax.plot(p, fresh_primary["q80"], linestyle=":", linewidth=1.8, label="80th percentile")
ax.set_xlabel(r"$p$")
ax.set_ylabel(r"Iterations to reach $R_t < 10^{-8}$")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper left")

ax = axes[1]
ax.plot(p, fresh_tol["T_1e_4"], marker="o", linewidth=1.5, markersize=3.5, label=r"$10^{-4}$")
ax.plot(p, fresh_tol["T_1e_6"], marker="s", linewidth=1.5, markersize=3.5, label=r"$10^{-6}$")
ax.plot(p, fresh_tol["T_1e_8"], marker="^", linewidth=1.5, markersize=4, label=r"$10^{-8}$")
ax.set_xlabel(r"$p$")
ax.set_ylabel("Median iteration count")
ax.grid(True, alpha=0.25)
ax.legend(title="Threshold", loc="upper left")

fig.tight_layout(pad=0.8, w_pad=2.0)
fig.savefig(REPRO_DIR / "A3_main_body_style.pdf", bbox_inches="tight")
fig.savefig(REPRO_DIR / "A3_main_body_style.svg", bbox_inches="tight")
plt.show()

print("saved reproduced figure in reproduced/")


## Optional post-run diagnostic

After the main A3 figure had been inspected, a separate descriptive diagnostic examined the widening high-$p$ variability. It did not enter the A3 paper figure or change the experiment. The strongest association was between iteration count and the smallest terminal block weight at $p=127.3$.


In [ ]:
from scipy.stats import spearmanr

hi = fresh_run[fresh_run["p"] == 127.3]
rho_min, _ = spearmanr(hi["T_1e_8"], hi["min_terminal_w"])
rho_kappa, _ = spearmanr(hi["T_1e_8"], hi["kappa_2_A"])
print(f"Spearman(T_1e-8, min terminal weight) at p=127.3: {rho_min:.6f}")
print(f"Spearman(T_1e-8, kappa_2(A)) at p=127.3: {rho_kappa:.6f}")
